# Notebook 10 — Spatial Risk Map

Generate a choropleth map showing predicted stranding counts per region for a target week.

**Reads:** `models/lgbm_model.pkl`, `plankton_imputed_lookup.parquet`, `final_dataset.parquet`  
**Writes:** `figures/risk_map_[YYYY-WW].png`


In [ ]:
import pandas as pd
import numpy as np
import pickle
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from datetime import datetime, timedelta

from se_coast_strandings.contextual_data.lunar_phases import moon_age, moon_phase
from se_coast_strandings.transformations import make_cyclic, make_cyclic_season
from se_coast_strandings import make_degrees

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
FIGURES_DIR = Path("../figures")
REFERENCE_DIR = Path("../data/reference")
FIGURES_DIR.mkdir(exist_ok=True)

# ── Region tuning ────────────────────────────────────────────────────────────
# Must match the DEGREES_PER_BAND used when building final_dataset.parquet
# (notebook 06) and plankton_imputed_lookup.parquet (notebook 05).
DEGREES_PER_BAND = 0.5
REGIONS = make_degrees(DEGREES_PER_BAND)

## Load Model and Data


In [ ]:
with open(MODELS_DIR / "lgbm_model.pkl", "rb") as f:
    model = pickle.load(f)

weekly = pd.read_parquet(PROCESSED_DIR / "final_dataset.parquet")
plankton = pd.read_parquet(PROCESSED_DIR / "plankton_imputed_lookup.parquet")
plankton = plankton.rename(columns={"ds": "week_start", "yhat": "plankton_density"})
plankton["week_start"] = pd.to_datetime(plankton["week_start"])

print(f"Model loaded. Historical data: {len(weekly)} rows")

## Set Target Week


In [ ]:
target_week = pd.Timestamp("2023-04-03")  # ISO 2023-W14

iso_year, iso_week, _ = target_week.isocalendar()
print(
    f"Target week: {target_week.strftime('%Y-%m-%d')} (ISO {iso_year}-W{iso_week:02d})"
)

## Build Feature Rows for Each Region


In [ ]:
regions = [label for label, _, _ in REGIONS]

# Get the feature columns the model expects
FEATURE_COLS = (
    model.feature_name()
    if hasattr(model, "feature_name")
    else [
        "month_sin",
        "month_cos",
        "dayofyear_sin",
        "dayofyear_cos",
        "season_sin",
        "season_cos",
        "moon_age",
        "temperature_2m_max_0_days_prior_mean",
        "temperature_2m_min_0_days_prior_mean",
        "temperature_2m_max_0_days_prior_max",
        "temp_delta_day0_mean",
        "plankton_density",
        "stranding_count_lag_1",
        "stranding_count_lag_52",
        *[f"region_{r}" for r in regions],
    ]
)

rows = []
for region in regions:
    row = {}

    # Cyclic time features
    month_s = pd.Series([target_week.month])
    doy_s = pd.Series([target_week.dayofyear])
    date_s = pd.Series([target_week])

    m_sin, m_cos = make_cyclic(month_s, 12, name="month")
    row["month_sin"] = float(m_sin.iloc[0])
    row["month_cos"] = float(m_cos.iloc[0])

    d_sin, d_cos = make_cyclic(doy_s, 365, name="dayofyear")
    row["dayofyear_sin"] = float(d_sin.iloc[0])
    row["dayofyear_cos"] = float(d_cos.iloc[0])

    s_sin, s_cos = make_cyclic_season(date_s, name="season")
    row["season_sin"] = float(s_sin.iloc[0])
    row["season_cos"] = float(s_cos.iloc[0])

    # Moon features
    row["moon_age"] = moon_age(target_week)

    # Plankton density (nearest week from lookup)
    region_plankton = plankton[plankton["region"] == region]
    if not region_plankton.empty:
        diffs = (region_plankton["week_start"] - target_week).abs()
        nearest_idx = diffs.idxmin()
        row["plankton_density"] = float(
            region_plankton.loc[nearest_idx, "plankton_density"]
        )
    else:
        row["plankton_density"] = 0.0

    # Weather: use last observed week as proxy (MVP)
    region_hist = weekly[weekly["region"] == region].sort_values("week_start")
    if not region_hist.empty:
        last_row = region_hist.iloc[-1]
        for col in [
            "temperature_2m_max_0_days_prior_mean",
            "temperature_2m_min_0_days_prior_mean",
            "temperature_2m_max_0_days_prior_max",
            "temp_delta_day0_mean",
        ]:
            if col in last_row.index:
                row[col] = float(last_row[col]) if pd.notna(last_row[col]) else 0.0

    # Lag features: use last known values
    if not region_hist.empty:
        row["stranding_count_lag_1"] = float(region_hist.iloc[-1]["stranding_count"])
        if len(region_hist) >= 52:
            row["stranding_count_lag_52"] = float(
                region_hist.iloc[-52]["stranding_count"]
            )
        else:
            row["stranding_count_lag_52"] = float(region_hist["stranding_count"].mean())

    # Region one-hot
    for r in regions:
        row[f"region_{r}"] = 1.0 if r == region else 0.0

    row["region"] = region
    rows.append(row)

feature_df = pd.DataFrame(rows)

# Ensure all expected columns exist
for col in FEATURE_COLS:
    if col not in feature_df.columns:
        feature_df[col] = 0.0

print(feature_df[["region"] + [c for c in FEATURE_COLS if c in feature_df.columns]])

## Predict


In [ ]:
predictions = model.predict(feature_df[FEATURE_COLS])
feature_df["predicted_count"] = predictions

print(f"\nPredicted strandings for week of {target_week.strftime('%Y-%m-%d')}:")
for _, row in feature_df.iterrows():
    print(f"  {row['region']:>10s}: {row['predicted_count']:.1f}")

## Generate Choropleth Risk Map


In [ ]:
import matplotlib.gridspec as gridspec

# Load state boundaries
states = gpd.read_file(REFERENCE_DIR / "cb_2018_us_state_5m.shp")
se_states = states[states["NAME"].isin(["Virginia", "North Carolina", "South Carolina"])]

# Build region polygons for predicted values
region_polys = []
for label, lat_min, lat_max in REGIONS:
    pred = feature_df[feature_df["region"] == label]["predicted_count"].values[0]
    actual_row = weekly[(weekly["week_start"] == target_week) & (weekly["region"] == label)]
    actual = float(actual_row["stranding_count"].values[0]) if not actual_row.empty else 0.0
    region_polys.append({
        "region": label,
        "lat_min": lat_min,
        "lat_max": lat_max,
        "lat_center": (lat_min + lat_max) / 2,
        "predicted": pred,
        "actual": actual,
    })

region_df = pd.DataFrame(region_polys)

vmax = max(region_df["predicted"].max(), region_df["actual"].max(), 1)
norm = plt.Normalize(vmin=0, vmax=vmax)
cmap = plt.cm.YlOrRd

panel_cfg = [
    ("Predicted", "predicted", "Predicted Strandings"),
    ("Actual",    "actual",    "Actual Strandings"),
]

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.10)

for col, (title, value_col, cbar_label) in enumerate(panel_cfg):
    ax = fig.add_subplot(gs[col])
    se_states.plot(ax=ax, edgecolor="black", facecolor="lightgray", linewidth=0.8)

    for _, row in region_df.iterrows():
        v = row[value_col]
        ax.axhspan(row["lat_min"], row["lat_max"], xmin=0, xmax=1,
                   alpha=0.45, color=cmap(norm(v)), zorder=2)
        val_str = f"{v:.1f}" if col == 0 else str(int(v))
        ax.text(-77.5, row["lat_center"], f"{row['region']}: {val_str}",
                ha="center", va="center", fontsize=12, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85), zorder=3)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation="horizontal", location="bottom",
                        shrink=0.75, pad=0.12)
    cbar.set_label(cbar_label, fontsize=13)
    cbar.ax.tick_params(labelsize=12)

    ax.set_xlim(-82, -74)
    ax.set_ylim(31.5, 38.5)
    ax.set_title(title, fontsize=16, fontweight="bold", pad=8)
    ax.set_xlabel("Longitude", fontsize=14, labelpad=6)
    ax.tick_params(axis="both", labelsize=13)
    if col == 0:
        ax.set_ylabel("Latitude", fontsize=14, labelpad=6)
    else:
        ax.set_yticklabels([])
        ax.set_ylabel("")

fig.suptitle(
    f"Marine Mammal Strandings — Week of {target_week.strftime('%Y-%m-%d')} (ISO {iso_year}-W{iso_week:02d})",
    fontsize=18, fontweight="bold"
)
plt.subplots_adjust(top=0.94)

out_path = FIGURES_DIR / f"risk_map_{iso_year}-W{iso_week:02d}.png"
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"\nSaved: {out_path}")
print(f"Total predicted: {region_df['predicted'].sum():.1f}  |  Total actual: {int(region_df['actual'].sum())}")

## April 2023 Retrospective — Predicted vs Actual

Run the model on the week of 2023-04-03 and compare to ground truth.

In [ ]:
# ── April 2023 retrospective: predicted map ──────────────────────────────────
retro_week = pd.Timestamp("2023-04-03")
iso_year_r, iso_week_r, _ = retro_week.isocalendar()

rows_r = []
for region in regions:
    row = {}

    month_s = pd.Series([retro_week.month])
    doy_s   = pd.Series([retro_week.dayofyear])
    date_s  = pd.Series([retro_week])

    m_sin, m_cos = make_cyclic(month_s, 12, name="month")
    row["month_sin"] = float(m_sin.iloc[0])
    row["month_cos"] = float(m_cos.iloc[0])

    d_sin, d_cos = make_cyclic(doy_s, 365, name="dayofyear")
    row["dayofyear_sin"] = float(d_sin.iloc[0])
    row["dayofyear_cos"] = float(d_cos.iloc[0])

    s_sin, s_cos = make_cyclic_season(date_s, name="season")
    row["season_sin"] = float(s_sin.iloc[0])
    row["season_cos"] = float(s_cos.iloc[0])

    row["moon_age"] = moon_age(retro_week)

    region_plankton = plankton[plankton["region"] == region]
    if not region_plankton.empty:
        diffs = (region_plankton["week_start"] - retro_week).abs()
        row["plankton_density"] = float(region_plankton.loc[diffs.idxmin(), "plankton_density"])
    else:
        row["plankton_density"] = 0.0

    region_hist = weekly[(weekly["region"] == region) & (weekly["week_start"] < retro_week)].sort_values("week_start")
    if not region_hist.empty:
        last_row = region_hist.iloc[-1]
        for col in [
            "temperature_2m_max_0_days_prior_mean",
            "temperature_2m_min_0_days_prior_mean",
            "temperature_2m_max_0_days_prior_max",
            "temp_delta_day0_mean",
        ]:
            row[col] = float(last_row[col]) if col in last_row.index and pd.notna(last_row[col]) else 0.0
        row["stranding_count_lag_1"]  = float(region_hist.iloc[-1]["stranding_count"])
        row["stranding_count_lag_52"] = float(region_hist.iloc[-52]["stranding_count"]) if len(region_hist) >= 52 else float(region_hist["stranding_count"].mean())

    for r in regions:
        row[f"region_{r}"] = 1.0 if r == region else 0.0

    row["region"] = region
    rows_r.append(row)

feature_df_r = pd.DataFrame(rows_r)
for col in FEATURE_COLS:
    if col not in feature_df_r.columns:
        feature_df_r[col] = 0.0

feature_df_r["predicted_count"] = model.predict(feature_df_r[FEATURE_COLS])

print(f"Predicted strandings for week of {retro_week.strftime('%Y-%m-%d')}:")
for _, row in feature_df_r.iterrows():
    print(f"  {row['region']:>10s}: {row['predicted_count']:.1f}")

# ── choropleth: predicted ────────────────────────────────────────────────────
region_polys_r = []
for label, lat_min, lat_max in REGIONS:
    pred = feature_df_r[feature_df_r["region"] == label]["predicted_count"].values[0]
    region_polys_r.append({"region": label, "lat_min": lat_min, "lat_max": lat_max,
                            "lat_center": (lat_min + lat_max) / 2, "value": pred})

region_df_r = pd.DataFrame(region_polys_r)

actual_vals = (
    weekly[(weekly["week_start"] == retro_week)]
    .set_index("region")["stranding_count"]
)
vmin_shared = 0
vmax_shared = max(region_df_r["value"].max(), actual_vals.max(), 1)
norm_shared = plt.Normalize(vmin=vmin_shared, vmax=vmax_shared)
cmap_shared = plt.cm.YlOrRd

fig, ax = plt.subplots(figsize=(12, 10))
se_states.plot(ax=ax, edgecolor="black", facecolor="lightgray", linewidth=0.8)

for _, row in region_df_r.iterrows():
    color = cmap_shared(norm_shared(row["value"]))
    ax.axhspan(row["lat_min"], row["lat_max"], xmin=0, xmax=1, alpha=0.4, color=color, zorder=2)
    ax.text(-77.5, row["lat_center"], f"{row['region']}: {row['value']:.1f}",
            ha="center", va="center", fontsize=16, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8), zorder=3)

sm = plt.cm.ScalarMappable(cmap=cmap_shared, norm=norm_shared)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, label="Predicted Weekly Strandings",
                    orientation="horizontal", location="bottom", shrink=0.6, pad=0.08)
cbar.set_label("Predicted Weekly Strandings", fontsize=14)
cbar.ax.tick_params(labelsize=12)

ax.set_xlim(-82, -74)
ax.set_ylim(31.5, 38.5)
ax.set_title(
    f"Model Predictions — Week of {retro_week.strftime('%Y-%m-%d')} (ISO {iso_year_r}-W{iso_week_r:02d})",
    fontsize=18, pad=15)
ax.set_xlabel("Longitude", fontsize=14)
ax.set_ylabel("Latitude", fontsize=14)
ax.tick_params(axis="both", labelsize=12)

out_path_pred = FIGURES_DIR / f"risk_map_predicted_{iso_year_r}-W{iso_week_r:02d}.png"
plt.tight_layout()
plt.savefig(out_path_pred, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path_pred}")

In [ ]:
# ── April 2023 retrospective: actual map ─────────────────────────────────────
actual_week = weekly[weekly["week_start"] == retro_week].copy()

region_polys_a = []
for label, lat_min, lat_max in REGIONS:
    row_data = actual_week[actual_week["region"] == label]
    count = float(row_data["stranding_count"].values[0]) if not row_data.empty else 0.0
    region_polys_a.append({"region": label, "lat_min": lat_min, "lat_max": lat_max,
                            "lat_center": (lat_min + lat_max) / 2, "value": count})

region_df_a = pd.DataFrame(region_polys_a)

fig, ax = plt.subplots(figsize=(12, 10))
se_states.plot(ax=ax, edgecolor="black", facecolor="lightgray", linewidth=0.8)

for _, row in region_df_a.iterrows():
    color = cmap_shared(norm_shared(row["value"]))
    ax.axhspan(row["lat_min"], row["lat_max"], xmin=0, xmax=1, alpha=0.4, color=color, zorder=2)
    ax.text(-77.5, row["lat_center"], f"{row['region']}: {int(row['value'])}",
            ha="center", va="center", fontsize=16, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8), zorder=3)

sm2 = plt.cm.ScalarMappable(cmap=cmap_shared, norm=norm_shared)
sm2.set_array([])
cbar2 = fig.colorbar(sm2, ax=ax, label="Actual Weekly Strandings",
                     orientation="horizontal", location="bottom", shrink=0.6, pad=0.08)
cbar2.set_label("Actual Weekly Strandings", fontsize=14)
cbar2.ax.tick_params(labelsize=12)

ax.set_xlim(-82, -74)
ax.set_ylim(31.5, 38.5)
ax.set_title(
    f"Actual Marine Mammal Strandings — Week of {retro_week.strftime('%Y-%m-%d')} (ISO {iso_year_r}-W{iso_week_r:02d})",
    fontsize=18, pad=15)
ax.set_xlabel("Longitude", fontsize=14)
ax.set_ylabel("Latitude", fontsize=14)
ax.tick_params(axis="both", labelsize=12)

out_path_actual = FIGURES_DIR / f"risk_map_actual_{iso_year_r}-W{iso_week_r:02d}.png"
plt.tight_layout()
plt.savefig(out_path_actual, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path_actual}")

In [ ]:
# ── Side-by-side: Predicted | Actual ────────────────────────────────────────
from matplotlib.colors import TwoSlopeNorm
import matplotlib.gridspec as gridspec

predicted = {r: feature_df_r[feature_df_r["region"] == r]["predicted_count"].values[0] for r in regions}
actual    = {r: float(weekly[(weekly["week_start"] == retro_week) & (weekly["region"] == r)]["stranding_count"].values[0]) for r in regions}
diff      = {r: predicted[r] - actual[r] for r in regions}

vmax_pa = max(max(predicted.values()), max(actual.values()), 1)
norm_pa = plt.Normalize(vmin=0, vmax=vmax_pa)
cmap_pa = plt.cm.YlOrRd

panel_cfg = [
    ("Predicted", predicted, "Predicted Strandings"),
    ("Actual",    actual,    "Actual Strandings"),
]

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.10)

for col, (panel_title, values, cbar_label) in enumerate(panel_cfg):
    ax = fig.add_subplot(gs[col])
    se_states.plot(ax=ax, edgecolor="black", facecolor="lightgray", linewidth=0.8)

    for label, lat_min, lat_max in REGIONS:
        v = values[label]
        ax.axhspan(lat_min, lat_max, xmin=0, xmax=1, alpha=0.45,
                   color=cmap_pa(norm_pa(v)), zorder=2)
        val_str = f"{v:.1f}" if col == 0 else str(int(v))
        ax.text(-77.5, (lat_min + lat_max) / 2, f"{label}: {val_str}",
                ha="center", va="center", fontsize=15, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85), zorder=3)

    sm = plt.cm.ScalarMappable(cmap=cmap_pa, norm=norm_pa)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation="horizontal", location="bottom",
                        shrink=0.75, pad=0.12)
    cbar.set_label(cbar_label, fontsize=13)
    cbar.ax.tick_params(labelsize=12)

    ax.set_xlim(-82, -74)
    ax.set_ylim(31.5, 38.5)
    ax.set_title(panel_title, fontsize=16, fontweight="bold", pad=8)
    ax.set_xlabel("Longitude", fontsize=14, labelpad=6)
    ax.tick_params(axis="both", labelsize=13)
    if col == 0:
        ax.set_ylabel("Latitude", fontsize=14, labelpad=6)
    else:
        ax.set_yticklabels([])
        ax.set_ylabel("")

fig.suptitle(
    f"Marine Mammal Strandings — Week of {retro_week.strftime('%m/%d/%Y')}",
    fontsize=18, fontweight="bold"
)
plt.subplots_adjust(top=0.97)
out_path_combined = FIGURES_DIR / f"risk_map_comparison_{iso_year_r}-W{iso_week_r:02d}.png"
plt.savefig(out_path_combined, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path_combined}")

In [ ]:
# ── Difference map: Predicted − Actual ───────────────────────────────────────
diff_abs_max = max(abs(v) for v in diff.values())
diff_abs_max = max(diff_abs_max, 0.5)
norm_diff = TwoSlopeNorm(vmin=-diff_abs_max, vcenter=0, vmax=diff_abs_max)
cmap_diff = plt.cm.RdYlGn_r

fig, ax = plt.subplots(figsize=(12, 10))
se_states.plot(ax=ax, edgecolor="black", facecolor="lightgray", linewidth=0.8)

for label, lat_min, lat_max in REGIONS:
    v = diff[label]
    ax.axhspan(lat_min, lat_max, xmin=0, xmax=1, alpha=0.45,
               color=cmap_diff(norm_diff(v)), zorder=2)
    val_str = f"+{v:.1f}" if v > 0 else f"{v:.1f}"
    ax.text(-77.5, (lat_min + lat_max) / 2, f"{label}: {val_str}",
            ha="center", va="center", fontsize=16, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85), zorder=3)

sm = plt.cm.ScalarMappable(cmap=cmap_diff, norm=norm_diff)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation="horizontal", location="bottom",
                    shrink=0.65, pad=0.06)
cbar.set_label("Difference (Predicted − Actual)", fontsize=13)
cbar.ax.tick_params(labelsize=12)

ax.set_xlim(-82, -74)
ax.set_ylim(31.5, 38.5)
ax.set_title(
    f"Prediction Error — Week of {retro_week.strftime('%m/%d/%Y')}",
    fontsize=18, fontweight="bold", pad=8
)
ax.set_xlabel("Longitude", fontsize=14, labelpad=6)
ax.set_ylabel("Latitude", fontsize=14, labelpad=6)
ax.tick_params(axis="both", labelsize=13)

out_path_diff = FIGURES_DIR / f"risk_map_difference_{iso_year_r}-W{iso_week_r:02d}.png"
plt.savefig(out_path_diff, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path_diff}")